# SFT Training: Balanced Thinking (9000 samples)

**Strategy:**
- **Single-Stage**: 9000 balanced data (1500/type × 6 types) with thinking CoT → learn reasoning patterns
- Stage 2 disabled

**Critical**: User prompt suffix matches official evaluation **EXACTLY**.

In [1]:
import subprocess, sys, os, glob

# --- Step 1: Install trl + datasets from our offline packages ---
# Use --no-deps to avoid dependency resolution issues in offline mode
offline_dirs = [
    "/kaggle/input/sft-offline-packages",
    "/kaggle/input/datasets/hastws/sft-offline-packages",
]
offline_dir = None
for d in offline_dirs:
    if os.path.isdir(d):
        whl_files = [f for f in os.listdir(d) if f.endswith('.whl')]
        if whl_files:
            offline_dir = d
            print(f"Found offline packages at: {d} ({len(whl_files)} wheels)")
            break

if offline_dir:
    # Install ALL wheels with --no-deps (system packages satisfy dependencies)
    whls = sorted(glob.glob(os.path.join(offline_dir, "*.whl")))
    print(f"Installing {len(whls)} wheels with --no-deps...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-deps"] + whls,
        capture_output=True, text=True, timeout=180
    )
    if result.returncode == 0:
        print("Installed all offline wheels")
    else:
        print(f"Install error: {result.stderr[-500:]}")
        # Try one by one
        for whl in whls:
            r = subprocess.run(
                [sys.executable, "-m", "pip", "install", "-q", "--no-deps", whl],
                capture_output=True, text=True, timeout=60
            )
            name = os.path.basename(whl).split('-')[0]
            if r.returncode == 0:
                print(f"  OK: {name}")
            else:
                print(f"  FAIL: {name}: {r.stderr[-100:]}")
else:
    print("No offline packages found!")
    if os.path.isdir("/kaggle/input"):
        for item in sorted(os.listdir("/kaggle/input")):
            print(f"  /kaggle/input/{item}/")

# Verify critical packages
for pkg in ["trl", "datasets"]:
    try:
        mod = __import__(pkg)
        print(f"  {pkg}={getattr(mod, '__version__', '?')}")
    except ImportError as e:
        raise RuntimeError(f"FATAL: {pkg} not importable after install: {e}")

# --- Step 2: flash_attn from dennisfong ---
try:
    import flash_attn
    print(f"flash_attn={flash_attn.__version__}")
except ImportError:
    fa_whls = glob.glob("/kaggle/input/**/*flash_attn*.whl", recursive=True)
    if fa_whls:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", fa_whls[0]],
                       capture_output=True, text=True, timeout=120)
        print(f"Installed flash_attn from {os.path.basename(fa_whls[0])}")
    else:
        print("flash_attn not available (slower attention)")

print("\nPackage setup complete")

Found offline packages at: /kaggle/input/datasets/hastws/sft-offline-packages (1 wheels)
Installing 1 wheels with --no-deps...
Installed all offline wheels
  trl=0.29.1
  datasets=4.0.0
flash_attn=2.8.3

Package setup complete


In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import sys
import stat
import shutil
import gc
import zipfile
import time
import json
import polars as pl
import pandas as pd
import torch
import torch.nn.functional as F
import kagglehub
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

print(f"torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    mem = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"GPU memory: {mem / 1024**3:.1f} GB")

# --- Triton / Kaggle environment fixes ---
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast:
        x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None:
        out = out + bias.float()
    if z is not None:
        out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

# Copy PTXAS binaries to writable temp
src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"
dst = "/tmp/ptxas-blackwell"
if os.path.exists(src):
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    import triton.backends.nvidia as nv_backend
    src_bin = os.path.join(os.path.dirname(nv_backend.__file__), "bin")
    dst_bin = "/tmp/triton_nvidia_bin"
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")
    os.environ["TRITON_PTXAS_PATH"] = dst
    print("✅ Triton patched")
else:
    print("⚠️ ptxas-blackwell not found, skipping Triton patch")

print("✅ Imports and environment ready")


/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)


torch: 2.12.0.dev20260324+cu128, CUDA: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
GPU memory: 95.0 GB
✅ Triton patched
✅ Imports and environment ready


## Configuration

All hyperparameters are defined here. The `PROMPT_SUFFIX` is copied verbatim from the **official evaluation metric script** — do not modify it.


In [3]:
# =============================================
#  🔧 HYPERPARAMETERS — EDIT HERE
# =============================================

# --- Stage 1: Thinking + boxed (single-stage) ---
STAGE1_N_SAMPLES = 2000   # 500/type × 4 types
STAGE1_THINKING_ONLY = False  # False = include all rows
STAGE1_ANSWER_ONLY = False   # False = use thinking format with \boxed{}
STAGE1_DROP_LONG = True    # True = drop samples exceeding STAGE1_MAX_SEQ (zero truncation)
STAGE1_LR        = 2e-4
STAGE1_EPOCHS    = 3
STAGE1_MAX_SEQ   = 512     # All samples fit within 512 tokens
STAGE1_BATCH     = 1       # Per-device batch size
STAGE1_GRAD_ACCUM = 8      # Effective batch = 1 × 8 = 8
STAGE1_PACKING   = True    # Safe with max_seq=512 + long samples dropped
# --- Stage 2: Format polish (disabled for answer-only) ---
STAGE2_ENABLED   = False
STAGE2_N_SAMPLES = 600
STAGE2_LR        = 4e-5    # 1/5 of Stage 1
STAGE2_EPOCHS    = 1
STAGE2_MAX_SEQ   = 512     # answer-only is short
STAGE2_BATCH     = 1
STAGE2_GRAD_ACCUM = 4
STAGE2_PACKING   = True

# --- LoRA ---
LORA_RANK        = 32
LORA_ALPHA       = 64
LORA_DROPOUT     = 0.05

# --- Type Filter (None = all 6 types) ---
TRAIN_TYPES      = ['numeral', 'gravity', 'unit_conv', 'cipher']    # 4 types

# --- Holdout Evaluation ---
HOLDOUT_ENABLED  = True
HOLDOUT_N_PER_TYPE = 20   # 20 per type × 6 = 120 total

# =============================================
#  🔴 OFFICIAL PROMPT SUFFIX — DO NOT MODIFY
# =============================================
# Source: competition_notebooks/nemotron-baseline-evaluation.ipynb
# Lines: user_content = item.prompt + '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

OUTPUT_DIR = "/kaggle/working/adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)
EVAL_MAX_TOKENS  = 512     # Max tokens for holdout eval (official=3584, reduced to save time)

print(f"Stage 1: n={STAGE1_N_SAMPLES}, answer_only={STAGE1_ANSWER_ONLY}, drop_long={STAGE1_DROP_LONG}")
print(f"Stage 1: LR={STAGE1_LR}, epochs={STAGE1_EPOCHS}, max_seq={STAGE1_MAX_SEQ}")
print(f"Stage 1: batch={STAGE1_BATCH}, grad_accum={STAGE1_GRAD_ACCUM}, packing={STAGE1_PACKING}")
print(f"Stage 2: enabled={STAGE2_ENABLED}, n={STAGE2_N_SAMPLES}, LR={STAGE2_LR}, packing={STAGE2_PACKING}")
print(f"LoRA: rank={LORA_RANK}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print(f"Train types: {TRAIN_TYPES if TRAIN_TYPES else 'ALL'}")
print(f"Holdout: enabled={HOLDOUT_ENABLED}, n_per_type={HOLDOUT_N_PER_TYPE}")
print(f"Eval max tokens: {EVAL_MAX_TOKENS}")
print(f"Prompt suffix: {repr(PROMPT_SUFFIX)}")

Stage 1: n=2000, answer_only=False, drop_long=True
Stage 1: LR=0.0002, epochs=3, max_seq=512
Stage 1: batch=1, grad_accum=8, packing=True
Stage 2: enabled=False, n=600, LR=4e-05, packing=True
LoRA: rank=32, alpha=64, dropout=0.05
Train types: ['numeral', 'gravity', 'unit_conv', 'cipher']
Holdout: enabled=True, n_per_type=20
Eval max tokens: 512
Prompt suffix: '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'


## Data Loading & Statistics

In [4]:
print("=== DATA LOADING: sft_thinking.csv (9000 balanced) ===")  # Version marker

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
COMP_DATA = '/kaggle/input/nvidia-nemotron-3-reasoning-challenge'

# --- Exhaustive search for training data ---
import glob as _glob

TARGET_FILE = 'sft_thinking.csv'

# Method 1: Check known candidates
_COT_CANDIDATES = [
    '/kaggle/input/prog-cot-training-data',
    '/kaggle/input/datasets/hastws/prog-cot-training-data',
]
COT_DATA = None
for d in _COT_CANDIDATES:
    csv_path = os.path.join(d, TARGET_FILE)
    if os.path.isfile(csv_path):
        COT_DATA = d
        print(f"Found via candidate: {csv_path}")
        break

# Method 2: Glob search under /kaggle/input/
if COT_DATA is None:
    print("Candidate dirs failed. Searching /kaggle/input/ recursively...")
    matches = _glob.glob(f'/kaggle/input/**/{TARGET_FILE}', recursive=True)
    if matches:
        COT_DATA = os.path.dirname(matches[0])
        print(f"Found via glob: {matches[0]}")
    else:
        print(f"ERROR: {TARGET_FILE} not found anywhere under /kaggle/input/")
        print("\n--- /kaggle/input/ contents ---")
        for root, dirs, files in os.walk('/kaggle/input/'):
            level = root.replace('/kaggle/input/', '').count(os.sep)
            indent = ' ' * 2 * level
            print(f'{indent}{os.path.basename(root)}/')
            if level < 3:
                subindent = ' ' * 2 * (level + 1)
                for f in files[:20]:
                    print(f'{subindent}{f}')
        raise FileNotFoundError(f"{TARGET_FILE} not found under /kaggle/input/. See listing above.")

print(f"COT_DATA = {COT_DATA}")
print(f"  Files: {os.listdir(COT_DATA)[:10]}")

# Load the merged dataset
train_df = pl.read_csv(f'{COT_DATA}/{TARGET_FILE}')

print(f"{'='*60}")
print(f"  Loaded: {TARGET_FILE} — {len(train_df)} rows")
print(f"{'='*60}")

# Statistics
pdf = train_df.to_pandas()
has_thinking = pdf['thinking'].fillna('').str.strip().str.len() > 0
n_with = has_thinking.sum()
n_without = (~has_thinking).sum()
print(f"\n  With thinking: {n_with} ({n_with/len(pdf)*100:.1f}%)")
print(f"  Without thinking (answer-only): {n_without} ({n_without/len(pdf)*100:.1f}%)")

# Classify thinking types
short_mask = has_thinking & (pdf['thinking'].str.len() < 50)
long_mask = has_thinking & (pdf['thinking'].str.len() >= 50)
print(f"  - Compact rules (<50 chars): {short_mask.sum()}")
print(f"  - Full CoT (≥50 chars): {long_mask.sum()}")

# Show thinking length distribution
print(f"\n  Thinking length stats (non-empty):")
lengths = pdf.loc[has_thinking, 'thinking'].str.len()
print(f"    min={lengths.min()}, median={lengths.median():.0f}, mean={lengths.mean():.0f}, max={lengths.max()}")

# Check for any data issues
print(f"\n  --- Sanity checks ---")
print(f"  Empty prompt: {(pdf['prompt'].fillna('').str.len() == 0).sum()}")
print(f"  Empty answer: {(pdf['answer'].fillna('').astype(str).str.len() == 0).sum()}")
boxed_in_thinking = pdf.loc[has_thinking, 'thinking'].str.contains(r'\\boxed', regex=True, na=False).sum()
print(f"  \\boxed in thinking: {boxed_in_thinking}")
print(f"  Columns: {list(pdf.columns)}")

# Type distribution
if 'type' in pdf.columns:
    print(f"\n  Type distribution:")
    for t in sorted(pdf['type'].unique()):
        print(f"    {t}: {(pdf['type']==t).sum()}")

# =============================================
#  TYPE INFERENCE FUNCTION
# =============================================
def _infer_type(prompt):
    p = prompt.lower()
    if 'bit manipulation' in p or '8-bit binary' in p:
        return 'bit_ops'
    elif 'numeral system' in p:
        return 'numeral'
    elif 'encrypt' in p or 'decrypt' in p:
        return 'cipher'
    elif 'gravitational' in p or 'gravity' in p or 'free-fall' in p:
        return 'gravity'
    elif 'unit' in p and ('convert' in p or 'conversion' in p):
        return 'unit_conv'
    elif 'symbol' in p or 'transformation rule' in p:
        return 'symbol'
    return 'unknown'

# =============================================
#  HOLDOUT SPLIT (for per-type evaluation)
# =============================================
if HOLDOUT_ENABLED:
    import re as _re
    
    # Load original train.csv for ground truth + type inference
    _orig_train = pl.read_csv(f'{COMP_DATA}/train.csv').to_pandas()
    _orig_train['type'] = _orig_train['prompt'].apply(_infer_type)
    
    # Stratified holdout: HOLDOUT_N_PER_TYPE per TRAIN_TYPE
    _holdout_types = TRAIN_TYPES if TRAIN_TYPES else sorted(_orig_train['type'].unique())
    _holdout_parts = []
    for _t in sorted(_holdout_types):
        _type_df = _orig_train[_orig_train['type'] == _t]
        _sampled = _type_df.sample(n=min(HOLDOUT_N_PER_TYPE, len(_type_df)), random_state=999)
        _holdout_parts.append(_sampled)
    holdout_df = pd.concat(_holdout_parts).reset_index(drop=True)
    holdout_ids = set(holdout_df['id'].tolist())
    
    print(f"\n{'='*60}")
    print(f"  HOLDOUT: {len(holdout_df)} samples ({HOLDOUT_N_PER_TYPE}/type)")
    print(f"{'='*60}")
    for _t in sorted(holdout_df['type'].unique()):
        print(f"    {_t}: {(holdout_df['type']==_t).sum()}")
    
    # Filter training data to exclude holdout
    _before = len(train_df)
    _keep_mask = ~train_df.to_pandas()['id'].isin(holdout_ids)
    train_df = pl.from_pandas(train_df.to_pandas()[_keep_mask.values].reset_index(drop=True))
    print(f"\n  Training data: {_before} → {len(train_df)} (removed {_before - len(train_df)} holdout overlap)")
else:
    holdout_df = None
    print("\nHoldout evaluation: DISABLED")

# =============================================
#  TYPE FILTER (train on subset of types)
# =============================================
if TRAIN_TYPES:
    _train_pdf = train_df.to_pandas()
    _train_pdf['_type'] = _train_pdf['prompt'].apply(_infer_type)
    _before_type = len(_train_pdf)
    _train_pdf = _train_pdf[_train_pdf['_type'].isin(TRAIN_TYPES)].reset_index(drop=True)
    
    print(f"\n{'='*60}")
    print(f"  TYPE FILTER: {TRAIN_TYPES}")
    print(f"  {_before_type} → {len(_train_pdf)} rows")
    print(f"{'='*60}")
    for _t in sorted(_train_pdf['_type'].unique()):
        print(f"    {_t}: {(_train_pdf['_type']==_t).sum()}")
    
    _train_pdf = _train_pdf.drop(columns=['_type'])
    train_df = pl.from_pandas(_train_pdf)
else:
    print("\nType filter: DISABLED (all types)")

=== DATA LOADING: sft_thinking.csv (9000 balanced) ===
Found via candidate: /kaggle/input/datasets/hastws/prog-cot-training-data/sft_thinking.csv
COT_DATA = /kaggle/input/datasets/hastws/prog-cot-training-data
  Files: ['sft_thinking.csv']
  Loaded: sft_thinking.csv — 9000 rows

  With thinking: 9000 (100.0%)
  Without thinking (answer-only): 0 (0.0%)
  - Compact rules (<50 chars): 0
  - Full CoT (≥50 chars): 9000

  Thinking length stats (non-empty):
    min=83, median=286, mean=282, max=613

  --- Sanity checks ---
  Empty prompt: 0
  Empty answer: 0
  \boxed in thinking: 0
  Columns: ['id', 'prompt', 'answer', 'thinking', 'type']

  Type distribution:
    bit_ops: 1500
    cipher: 1500
    gravity: 1500
    numeral: 1500
    symbol: 1500
    unit_conv: 1500

  HOLDOUT: 80 samples (20/type)
    cipher: 20
    gravity: 20
    numeral: 20
    unit_conv: 20

  Training data: 9000 → 8924 (removed 76 holdout overlap)

  TYPE FILTER: ['numeral', 'gravity', 'unit_conv', 'cipher']
  8924 → 5

## Prompt Format & Training Text

**Single-Stage Format:**
- Prompt suffix + thinking + `\boxed{}` answer all in one stage

The suffix must match the official evaluation metric script EXACTLY:
```python
user_content = item.prompt + '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
```

In [5]:
import math

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# =============================================
#  Prompt suffix verification (for Stage 2)
# =============================================
print("=== PROMPT SUFFIX VERIFICATION ===")
print(f"Our PROMPT_SUFFIX: {repr(PROMPT_SUFFIX)}")

official_suffix = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
assert PROMPT_SUFFIX == official_suffix, f"❌ MISMATCH!\nOurs:     {repr(PROMPT_SUFFIX)}\nOfficial: {repr(official_suffix)}"
print("✅ Prompt suffix matches official evaluation exactly")

# Show what official eval generates for inference
sample_prompt = "What is 2 + 2?"
official_inference_prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": sample_prompt + PROMPT_SUFFIX}],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)
print(f"\nOfficial inference prompt (enable_thinking=True):\n---\n{official_inference_prompt}\n---")

# =============================================
#  Helper: check if thinking is empty/missing
# =============================================
def _has_thinking(thinking):
    """Return True if thinking contains actual content (not NaN/None/empty)."""
    if thinking is None:
        return False
    if isinstance(thinking, float) and math.isnan(thinking):
        return False
    s = str(thinking).strip()
    return len(s) > 0 and s.lower() != 'nan'

# =============================================
#  Stage 1 builder: NO boxed, pure reasoning
# =============================================
def build_stage1_text(example):
    """Stage 1: Build training text. Answer-only or thinking+boxed mode."""
    prompt = example["prompt"]
    answer = str(example["answer"])
    
    if STAGE1_ANSWER_ONLY:
        # Answer-only: match inference format (prompt suffix + \boxed{})
        user_msg = prompt + PROMPT_SUFFIX
        text = (
            f"<|im_start|>user\n{user_msg}<|im_end|>\n"
            f"<|im_start|>assistant\n<think></think>\\boxed{{{answer}}}<|im_end|>"
        )
    else:
        # Single-stage: thinking + boxed + suffix all together
        user_msg = prompt + PROMPT_SUFFIX
        thinking = example.get("thinking", None)
        if _has_thinking(thinking):
            text = (
                f"<|im_start|>user\n{user_msg}<|im_end|>\n"
                f"<|im_start|>assistant\n<think>\n{str(thinking).strip()}\n</think>\n\\boxed{{{answer}}}<|im_end|>"
            )
        else:
            text = (
                f"<|im_start|>user\n{user_msg}<|im_end|>\n"
                f"<|im_start|>assistant\n<think></think>\\boxed{{{answer}}}<|im_end|>"
            )
    return {"text": text}

# =============================================
#  Stage 2 builder: thinking + boxed (thinking masked in loss)
# =============================================
THINK_END_TOKEN_ID = 13  # </think> token in Nemotron tokenizer

def build_stage2_text(example):
    """Stage 2: Unified format with thinking + \\boxed{}.
    Thinking content will be masked in loss computation."""
    prompt = example["prompt"]
    answer = str(example["answer"])
    thinking = example.get("thinking", "")
    user_msg = prompt + PROMPT_SUFFIX
    
    if _has_thinking(thinking):
        text = (
            f"<|im_start|>user\n{user_msg}<|im_end|>\n"
            f"<|im_start|>assistant\n<think>\n{str(thinking).strip()}\n</think>\n\\boxed{{{answer}}}<|im_end|>"
        )
    else:
        text = (
            f"<|im_start|>user\n{user_msg}<|im_end|>\n"
            f"<|im_start|>assistant\n<think></think>\\boxed{{{answer}}}<|im_end|>"
        )
    return {"text": text}

def tokenize_and_mask_thinking(example):
    """Tokenize and mask everything up to </think> (inclusive) in labels."""
    text = example['text']
    encoded = tokenizer(text, add_special_tokens=False, truncation=True, max_length=STAGE2_MAX_SEQ)
    input_ids = encoded['input_ids']
    labels = list(input_ids)
    
    # Find </think> token and mask everything up to and including it
    try:
        think_end_pos = input_ids.index(THINK_END_TOKEN_ID)
        for i in range(think_end_pos + 1):
            labels[i] = -100
    except ValueError:
        pass  # no </think> found, don't mask
    
    return {
        'input_ids': input_ids,
        'attention_mask': [1] * len(input_ids),
        'labels': labels,
    }

print("\n=== STAGE 1 FORMAT VERIFICATION ===")
sample_df = train_df.to_pandas()

if STAGE1_ANSWER_ONLY:
    # Test answer-only format
    row = sample_df.iloc[0].to_dict()
    result = build_stage1_text(row)
    text = result['text']
    print(f"\n--- STAGE 1: ANSWER-ONLY MODE (id={row['id']}) ---")
    print(text[:500])
    assert '<think></think>' in text, "Missing empty think tags"
    assert '\\boxed{' in text, "❌ Answer-only should contain \\boxed{}!"
    assert PROMPT_SUFFIX.lstrip('\n') in text, "❌ Missing prompt suffix!"
    print("✅ Answer-only: has suffix, has \\boxed{}, empty think")
else:
    # Test single-stage thinking+boxed format
    think_rows = sample_df[sample_df['thinking'].apply(_has_thinking)]
    if len(think_rows) > 0:
        row = think_rows.iloc[0].to_dict()
        result = build_stage1_text(row)
        text = result['text']
        print(f"\n--- STAGE 1: THINKING+BOXED (id={row['id']}) ---")
        print(text[:800])
        if len(text) > 800:
            print(f"... ({len(text)} chars total)")
        assert '<think>\n' in text, "Missing <think> tag"
        assert '\n</think>\n' in text, "Missing </think> tag"
        assert '\\boxed{' in text, "Missing \\boxed{} in thinking+boxed mode!"
        assert PROMPT_SUFFIX.lstrip('\n') in text, "Missing prompt suffix!"
        print("✅ Thinking+boxed: has suffix, has \\boxed{}, has thinking")

    # Test with an answer-only row (fallback format)
    ao_rows = sample_df[~sample_df['thinking'].apply(_has_thinking)]
    if len(ao_rows) > 0:
        row = ao_rows.iloc[0].to_dict()
        result = build_stage1_text(row)
        text = result['text']
        print(f"\n--- STAGE 1: NO-THINKING FALLBACK (id={row['id']}) ---")
        print(text[:500])
        assert '\\boxed{' in text, "Missing \\boxed{} in fallback!"
        assert '<think></think>' in text, "Missing empty think tags"
        print("✅ Fallback: has \\boxed{}, empty think")

print("\n=== STAGE 2 FORMAT VERIFICATION ===")
if not STAGE1_ANSWER_ONLY and len(think_rows) > 0:
    row = think_rows.iloc[0].to_dict()
    result = build_stage2_text(row)
    text = result['text']
    print(f"\n--- STAGE 2: THINKING ROW (id={row['id']}) ---")
    print(text[:500])
    if len(text) > 500:
        print(f"... ({len(text)} chars total)")
    assert PROMPT_SUFFIX.lstrip('\n') in text, "Missing prompt suffix"
    assert '\\boxed{' in text, "Missing \\boxed{}"
    assert '<think>\n' in text, "Missing thinking content"
    print("✅ Stage 2 row: has suffix, has boxed, has thinking")
    
    # Verify masking
    enc = tokenizer(text, add_special_tokens=False)
    ids = enc['input_ids']
    try:
        pos = ids.index(THINK_END_TOKEN_ID)
        total_tokens = len(ids)
        masked_tokens = pos + 1
        trained_tokens = total_tokens - masked_tokens
        print(f"  Masking: {masked_tokens}/{total_tokens} tokens masked, {trained_tokens} tokens trained")
    except ValueError:
        print("  WARNING: </think> token not found!")
else:
    print("  (Skipped - answer-only mode, no Stage 2 format to verify)")

print("\n✅ All format checks passed!")

=== PROMPT SUFFIX VERIFICATION ===
Our PROMPT_SUFFIX: '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
✅ Prompt suffix matches official evaluation exactly

Official inference prompt (enable_thinking=True):
---
<|im_start|>system
<|im_end|>
<|im_start|>user
What is 2 + 2?
Please put your final answer inside `\boxed{}`. For example: `\boxed{your answer}`<|im_end|>
<|im_start|>assistant
<think>

---

=== STAGE 1 FORMAT VERIFICATION ===

--- STAGE 1: THINKING+BOXED (id=d3926533) ---
<|im_start|>user
In Alice's Wonderland, numbers are secretly converted into a different numeral system. Some examples are given below:
57 -> LVII
45 -> XLV
78 -> LXXVIII
77 -> LXXVII
65 -> LXV
Now, write the number 43 in the Wonderland numeral system.
Please put your final answer inside `\boxed{}`. For example: `\boxed{your answer}`<|im_end|>
<|im_start|>assistant
<think>
The examples show Arabic to Roman numeral conversion.

Converting 43:
  40 fits into 43 → write 'XL',

In [6]:
# Convert dataset for Stage 1 (no boxed)
stage1_pdf = train_df.to_pandas()

# Filter by thinking mode
if STAGE1_THINKING_ONLY:
    before = len(stage1_pdf)
    stage1_pdf = stage1_pdf[stage1_pdf['thinking'].apply(_has_thinking)].reset_index(drop=True)
    print(f"Stage 1 thinking-only filter: {before} → {len(stage1_pdf)} rows (dropped {before - len(stage1_pdf)} answer-only)")
elif STAGE1_ANSWER_ONLY:
    print(f"Stage 1 answer-only mode: using all {len(stage1_pdf)} rows (thinking ignored)")

if STAGE1_N_SAMPLES is not None:
    stage1_pdf = stage1_pdf.sample(n=min(STAGE1_N_SAMPLES, len(stage1_pdf)), random_state=42)
    print(f"Stage 1 subsampled: {len(stage1_pdf)} rows")

hf_dataset = Dataset.from_pandas(stage1_pdf)
hf_dataset = hf_dataset.map(
    build_stage1_text,
    remove_columns=hf_dataset.column_names,
)
print(f"Stage 1 dataset formatted: {len(hf_dataset)} rows (no boxed)")

# Token length analysis
print("\n=== TOKEN LENGTH ANALYSIS ===")
import numpy as np

token_lengths = []
for i in range(len(hf_dataset)):
    toks = tokenizer(hf_dataset[i]['text'], add_special_tokens=False)
    token_lengths.append(len(toks['input_ids']))

tl = np.array(token_lengths)
print(f"  Total samples: {len(tl)}")
print(f"  Min tokens:    {tl.min()}")
print(f"  Median tokens: {np.median(tl):.0f}")
print(f"  Mean tokens:   {tl.mean():.0f}")
print(f"  P95 tokens:    {np.percentile(tl, 95):.0f}")
print(f"  P99 tokens:    {np.percentile(tl, 99):.0f}")
print(f"  Max tokens:    {tl.max()}")

# Drop samples exceeding max_seq if enabled
if STAGE1_DROP_LONG:
    keep_mask = tl <= STAGE1_MAX_SEQ
    n_drop = (~keep_mask).sum()
    print(f"\n  Dropping {n_drop} samples > {STAGE1_MAX_SEQ} tokens ({n_drop/len(tl)*100:.1f}%)")
    keep_indices = np.where(keep_mask)[0].tolist()
    hf_dataset = hf_dataset.select(keep_indices)
    tl = tl[keep_mask]
    print(f"  Remaining: {len(hf_dataset)} samples (max={tl.max()} tokens, zero truncation)")
else:
    truncated = (tl > STAGE1_MAX_SEQ).sum()
    print(f"\n  Truncated at {STAGE1_MAX_SEQ}: {truncated} ({truncated/len(tl)*100:.1f}%)")

# Show 3 samples: short, medium, long
sorted_idx = np.argsort(tl)
for label, idx in [("SHORTEST", sorted_idx[0]), ("MEDIAN", sorted_idx[len(sorted_idx)//2]), ("LONGEST", sorted_idx[-1])]:
    print(f"\n--- {label} ({tl[idx]} tokens) ---")
    print(hf_dataset[int(idx)]['text'][:400])

    if len(hf_dataset[int(idx)]['text']) > 400:
        print(f"... [truncated, {len(hf_dataset[int(idx)]['text'])} chars]")

Stage 1 subsampled: 2000 rows


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Stage 1 dataset formatted: 2000 rows (no boxed)

=== TOKEN LENGTH ANALYSIS ===
  Total samples: 2000
  Min tokens:    126
  Median tokens: 291
  Mean tokens:   300
  P95 tokens:    502
  P99 tokens:    511
  Max tokens:    520

  Dropping 16 samples > 512 tokens (0.8%)
  Remaining: 1984 samples (max=511 tokens, zero truncation)

--- SHORTEST (126 tokens) ---
<|im_start|>user
In Alice's Wonderland, numbers are secretly converted into a different numeral system. Some examples are given below:
36 -> XXXVI
9 -> IX
5 -> V
Now, write the number 4 in the Wonderland numeral system.
Please put your final answer inside `\boxed{}`. For example: `\boxed{your answer}`<|im_end|>
<|im_start|>assistant
<think>
The examples show Arabic to Roman numeral conversion.

Co
... [truncated, 496 chars]

--- MEDIAN (290 tokens) ---
<|im_start|>user
In Alice's Wonderland, secret encryption rules are used on text. Here are some examples:
zlwx qppq irp vamlpai xjjw -> bird sees the ancient door
feppa cjeax irwjebr

## Model Loading & LoRA Setup

In [7]:
# Mock cutlass/mamba3 to prevent import errors
from unittest.mock import MagicMock

_mock_modules = [
    "cutlass", "cutlass.cute", "cutlass.utils",
    "mamba_ssm.ops.cute", "mamba_ssm.ops.cute.mamba3",
    "mamba_ssm.ops.cute.mamba3.mamba3_step_fn",
    "mamba_ssm.ops.tilelang", "mamba_ssm.ops.tilelang.mamba3",
    "mamba_ssm.ops.tilelang.mamba3.mamba3_mimo",
]
for mod_name in _mock_modules:
    if mod_name not in sys.modules:
        sys.modules[mod_name] = MagicMock()
print(f"Mock modules injected: {len(_mock_modules)}")

# Load model
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16,
)
print(f"Model loaded in {time.time()-t0:.1f}s. Vocab size: {len(tokenizer)}")

# Patch: force slow path to bypass broken CUDA kernels
for name, mod in sys.modules.items():
    if "modeling_nemotron_h" in name:
        mod.is_fast_path_available = False
        print(f"Patched {name}: is_fast_path_available = False")

# Setup LoRA
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules="all-linear",
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Log model architecture summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal params: {total_params/1e9:.2f}B")
print(f"Trainable params: {trainable_params/1e6:.1f}M ({trainable_params/total_params*100:.2f}%)")


Mock modules injected: 9


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

Model loaded in 466.2s. Vocab size: 131072
Patched transformers_modules._1.modeling_nemotron_h: is_fast_path_available = False


/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:122: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:195: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_scaling_utils.py:90: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_linear.py:60: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_

trainable params: 883,873,792 || all params: 32,461,811,136 || trainable%: 2.7228

Total params: 32.46B
Trainable params: 883.9M (2.72%)


## Stage 1: Learn Reasoning

Train on 9000 balanced samples (1500/type × 6 types). This teaches the model to:
1. Recognize different problem types
2. Apply reasoning strategies within `<think>` tags  
3. Output answers in `\boxed{}` format

In [8]:
import triton.backends.nvidia.compiler as nv_compiler
os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"
nv_compiler.get_ptxas_version = lambda arch: "12.0"

eff_batch = STAGE1_BATCH * STAGE1_GRAD_ACCUM
total_steps = (len(hf_dataset) // eff_batch) * STAGE1_EPOCHS
print(f"{'='*60}")
print(f"  STAGE 1: Reasoning Training")
print(f"  Samples: {len(hf_dataset)}, LR: {STAGE1_LR}, Epochs: {STAGE1_EPOCHS}")
print(f"  Max Seq: {STAGE1_MAX_SEQ}, Batch: {STAGE1_BATCH}, Grad Accum: {STAGE1_GRAD_ACCUM}")
print(f"  Packing: {STAGE1_PACKING}, Effective batch: {eff_batch}")
print(f"  Estimated steps: ~{total_steps}")
print(f"{'='*60}")

stage1_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=STAGE1_BATCH,
    gradient_accumulation_steps=STAGE1_GRAD_ACCUM,
    num_train_epochs=STAGE1_EPOCHS,
    learning_rate=STAGE1_LR,
    logging_steps=10,
    bf16=True,
    max_grad_norm=1.0,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    save_strategy="no",
    report_to="none",
    dataset_text_field="text",
    max_length=STAGE1_MAX_SEQ,
    packing=STAGE1_PACKING,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": True},
)

stage1_trainer = SFTTrainer(
    model=model,
    train_dataset=hf_dataset,
    processing_class=tokenizer,
    args=stage1_args,
)

t0 = time.time()
stage1_result = stage1_trainer.train()
stage1_time = time.time() - t0

print(f"\n{'='*60}")
print(f"  STAGE 1 COMPLETE")
print(f"  Time: {stage1_time/60:.1f} min")
print(f"  Final loss: {stage1_result.training_loss:.4f}")
print(f"{'='*60}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


  STAGE 1: Reasoning Training
  Samples: 1984, LR: 0.0002, Epochs: 3
  Max Seq: 512, Batch: 1, Grad Accum: 8
  Packing: True, Effective batch: 8
  Estimated steps: ~744


Adding EOS to train dataset:   0%|          | 0/1984 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1984 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/1984 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 11, 'pad_token_id': 11}.


Step,Training Loss
10,14.401031
20,10.165353
30,5.584974
40,4.940305
50,4.393991
60,3.604924
70,4.042142
80,3.883897
90,3.654296
100,4.016759



  STAGE 1 COMPLETE
  Time: 233.6 min
  Final loss: 3.8143


## Stage 2: Format Polish (Optional)

Light fine-tuning with answer-only data at low LR. Reinforces `\boxed{}` output format without disrupting learned reasoning.

Set `STAGE2_ENABLED = False` above to skip this stage.


In [9]:
if STAGE2_ENABLED:
    from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
    
    print(f"{'='*60}")
    print(f"  STAGE 2: Format Polish (thinking masked)")
    print(f"{'='*60}")
    
    # Prepare Stage 2 dataset from THINKING rows (not answer-only)
    full_df = train_df.to_pandas()
    think_mask = full_df['thinking'].apply(_has_thinking)
    thinking_df = full_df[think_mask]
    
    n_sample = min(STAGE2_N_SAMPLES, len(thinking_df))
    stage2_df = thinking_df.sample(n=n_sample, random_state=42)
    
    eff_batch2 = STAGE2_BATCH * STAGE2_GRAD_ACCUM
    total_steps2 = (n_sample // eff_batch2) * STAGE2_EPOCHS
    print(f"  Thinking pool: {len(thinking_df)}")
    print(f"  Sampled for Stage 2: {n_sample}")
    print(f"  LR: {STAGE2_LR}, Epochs: {STAGE2_EPOCHS}, Max Seq: {STAGE2_MAX_SEQ}")
    print(f"  Batch: {STAGE2_BATCH}, Grad Accum: {STAGE2_GRAD_ACCUM}")
    print(f"  Loss: thinking MASKED, only \\boxed{{answer}} trained")
    print(f"  Estimated steps: ~{total_steps2}")
    
    # Build text then tokenize with thinking masked
    stage2_raw = Dataset.from_pandas(stage2_df)
    stage2_raw = stage2_raw.map(
        build_stage2_text,
        remove_columns=stage2_raw.column_names,
    )
    stage2_tokenized = stage2_raw.map(
        tokenize_and_mask_thinking,
        remove_columns=['text'],
    )
    
    # Show a Stage 2 sample with masking info
    print(f"\n--- Stage 2 sample ---")
    sample_ids = stage2_tokenized[0]['input_ids']
    sample_labels = stage2_tokenized[0]['labels']
    n_masked = sum(1 for l in sample_labels if l == -100)
    n_trained = sum(1 for l in sample_labels if l != -100)
    print(f"  Tokens: {len(sample_ids)}, masked: {n_masked}, trained: {n_trained}")
    # Decode the trained portion
    trained_ids = [i for i, l in zip(sample_ids, sample_labels) if l != -100]
    print(f"  Trained text: {tokenizer.decode(trained_ids)[:200]}")
    
    # Use Trainer (not SFTTrainer) for pre-tokenized data with custom labels
    collator = DataCollatorForSeq2Seq(tokenizer, padding=True)
    
    stage2_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=STAGE2_BATCH,
        gradient_accumulation_steps=STAGE2_GRAD_ACCUM,
        num_train_epochs=STAGE2_EPOCHS,
        learning_rate=STAGE2_LR,
        logging_steps=5,
        bf16=True,
        max_grad_norm=1.0,
        optim="adamw_torch",
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        save_strategy="no",
        report_to="none",
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": True},
    )
    
    stage2_trainer = Trainer(
        model=model,
        train_dataset=stage2_tokenized,
        data_collator=collator,
        args=stage2_args,
    )
    
    t0 = time.time()
    stage2_result = stage2_trainer.train()
    stage2_time = time.time() - t0
    
    print(f"\n{'='*60}")
    print(f"  STAGE 2 COMPLETE")
    print(f"  Time: {stage2_time/60:.1f} min")
    print(f"  Final loss: {stage2_result.training_loss:.4f}")
    print(f"{'='*60}")
else:
    print("Stage 2 SKIPPED (STAGE2_ENABLED=False)")

Stage 2 SKIPPED (STAGE2_ENABLED=False)


## Save & Package Submission

**IMPORTANT**: Save runs BEFORE holdout evaluation so the adapter is preserved even if eval OOM's.

In [10]:
# Save adapter weights and config
model.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to {OUTPUT_DIR}")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f:40s} {size/1024:.1f} KB")

# --- Override adapter_config.json with explicit config ---
# This ensures reproducibility and lets you tweak parameters directly here
ADAPTER_CONFIG = {
    "alpha_pattern": {},
    "auto_mapping": None,
    "base_model_name_or_path": MODEL_PATH,
    "bias": "none",
    "fan_in_fan_out": False,
    "inference_mode": True,
    "init_lora_weights": True,
    "layers_pattern": None,
    "layers_to_transform": None,
    "loftq_config": {},
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "modules_to_save": None,
    "peft_type": "LORA",
    "r": LORA_RANK,
    "rank_pattern": {},
    "revision": None,
    "target_modules": [
        "in_proj", "out_proj",
        "k_proj", "q_proj", "v_proj", "o_proj",
        "up_proj", "down_proj"
    ],
    "task_type": "CAUSAL_LM",
    "use_dora": False,
    "use_rslora": False,
}

config_path = os.path.join(OUTPUT_DIR, "adapter_config.json")
with open(config_path, "w") as f:
    json.dump(ADAPTER_CONFIG, f, indent=2)
print(f"\n✅ adapter_config.json overwritten with explicit config")
print(f"  r={ADAPTER_CONFIG['r']}, alpha={ADAPTER_CONFIG['lora_alpha']}, targets={ADAPTER_CONFIG['target_modules']}")

# Package into submission.zip
zip_path = "/kaggle/working/submission.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in ["adapter_model.safetensors", "adapter_config.json"]:
        fpath = os.path.join(OUTPUT_DIR, fname)
        if os.path.isfile(fpath):
            zf.write(fpath, fname)

print(f"\nCreated {zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB)")

# Verify submission
with zipfile.ZipFile(zip_path, 'r') as zf:
    names = zf.namelist()
    print(f"Contents: {names}")
    assert "adapter_config.json" in names, "❌ Missing adapter_config.json!"
    assert "adapter_model.safetensors" in names, "❌ Missing adapter_model.safetensors!"
    
    with zf.open("adapter_config.json") as cf:
        config = json.loads(cf.read())
        print(f"\nAdapter config in zip:")
        print(f"  r (rank):     {config.get('r', '?')}")
        print(f"  lora_alpha:   {config.get('lora_alpha', '?')}")
        print(f"  target_modules: {config.get('target_modules', '?')}")

print("\n✅ submission.zip is valid and ready to submit!")

Adapter saved to /kaggle/working/adapter
  README.md                                5.2 KB
  adapter_config.json                      1.1 KB
  adapter_model.safetensors                3454393.7 KB

✅ adapter_config.json overwritten with explicit config
  r=32, alpha=64, targets=['in_proj', 'out_proj', 'k_proj', 'q_proj', 'v_proj', 'o_proj', 'up_proj', 'down_proj']

Created /kaggle/working/submission.zip (3087.8 MB)
Contents: ['adapter_model.safetensors', 'adapter_config.json']

Adapter config in zip:
  r (rank):     32
  lora_alpha:   64
  target_modules: ['in_proj', 'out_proj', 'k_proj', 'q_proj', 'v_proj', 'o_proj', 'up_proj', 'down_proj']

✅ submission.zip is valid and ready to submit!


In [11]:
# =============================================
#  HOLDOUT EVALUATION — Per-Type Accuracy
# =============================================
import re as _re

eval_time = 0
results_df = None

# Save training metrics before cleanup
_s1_loss = stage1_result.training_loss if 'stage1_result' in dir() else -1
_s2_loss = stage2_result.training_loss if (STAGE2_ENABLED and 'stage2_result' in dir()) else -1

if HOLDOUT_ENABLED and holdout_df is not None:

    # --- Memory cleanup: free training state before inference ---
    print("Freeing training memory...")
    _g = globals()
    for _obj_name in ['stage1_trainer', 'stage2_trainer', 'stage1_result', 'stage2_result',
                       'hf_dataset', 'stage2_tokenized', 'stage2_raw', 'collator',
                       'stage1_args', 'stage2_args']:
        if _obj_name in _g:
            del _g[_obj_name]
    gc.collect()
    torch.cuda.empty_cache()
    _mem_free = torch.cuda.mem_get_info()[0] / 1024**3
    print(f"GPU free memory after cleanup: {_mem_free:.1f} GB")

    # --- Merge LoRA into base model to save ~8GB VRAM ---
    print("Merging LoRA into base model...")
    model = model.merge_and_unload()
    gc.collect()
    torch.cuda.empty_cache()
    _mem_free2 = torch.cuda.mem_get_info()[0] / 1024**3
    print(f"GPU free memory after merge: {_mem_free2:.1f} GB (saved {_mem_free2 - _mem_free:.1f} GB)")

    print(f"Holdout types: {sorted(holdout_df['type'].unique().tolist())}")

    # --- Official answer extraction (from nemotron-baseline-evaluation.ipynb) ---
    def extract_final_answer(text):
        if text is None:
            return 'NOT_FOUND'
        matches = _re.findall(r'\\boxed\{([^}]*)(?:\}|$)', text)
        if matches:
            non_empty = [m.strip() for m in matches if m.strip()]
            if non_empty:
                return non_empty[-1]
            return matches[-1].strip()
        patterns = [
            r'The final answer is:\s*([^\n]+)',
            r'Final answer is:\s*([^\n]+)',
            r'Final answer\s*[:：]\s*([^\n]+)',
            r'final answer\s*[:：]\s*([^\n]+)',
        ]
        for pattern in patterns:
            matches = _re.findall(pattern, text, _re.IGNORECASE)
            if matches:
                return matches[-1].strip()
        matches = _re.findall(r'-?\d+(?:\.\d+)?', text)
        if matches:
            return matches[-1]
        lines = [line.strip() for line in text.splitlines() if line.strip()]
        return lines[-1] if lines else 'NOT_FOUND'

    # --- Official verify (from nemotron-baseline-evaluation.ipynb) ---
    import math as _math
    def eval_verify(stored_answer, predicted):
        stored_answer = str(stored_answer).strip()
        predicted = str(predicted).strip()
        try:
            stored_num = float(stored_answer)
            predicted_num = float(predicted)
            return _math.isclose(stored_num, predicted_num, rel_tol=1e-2, abs_tol=1e-5)
        except Exception:
            return predicted.lower() == stored_answer.lower()

    EVAL_MAX_TOKENS = EVAL_MAX_TOKENS if 'EVAL_MAX_TOKENS' in dir() else 1024

    eval_results = []
    t0_eval = time.time()

    print(f"\n{'='*60}")
    print(f"  HOLDOUT EVAL: {len(holdout_df)} samples")
    print(f"  Types: {sorted(holdout_df['type'].unique().tolist())}")
    print(f"  Decoding: greedy (temperature=0.0) -- matches official eval")
    print(f"  Max new tokens: {EVAL_MAX_TOKENS}")
    print(f"{'='*60}")

    try:
        for eval_idx, row in holdout_df.iterrows():
            prompt = row['prompt']
            answer = str(row['answer'])
            qtype = row['type']

            # Build prompt exactly like official eval
            user_content = prompt + PROMPT_SUFFIX
            chat_prompt = tokenizer.apply_chat_template(
                [{"role": "user", "content": user_content}],
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=True,
            )

            inputs = tokenizer(chat_prompt, return_tensors="pt").to(model.device)
            prompt_len = inputs['input_ids'].shape[1]

            with torch.no_grad():
                output = model.generate(
                    **inputs,
                    max_new_tokens=EVAL_MAX_TOKENS,
                    do_sample=False,
                )

            gen_ids = output[0][prompt_len:]
            generated = tokenizer.decode(gen_ids, skip_special_tokens=False)
            predicted = extract_final_answer(generated)
            correct = eval_verify(answer, predicted)

            # --- Verbose per-sample output ---
            n_done = len(eval_results) + 1
            status = "✅" if correct else "❌"
            print(f"\n{'='*60}")
            print(f"  [{n_done}/{len(holdout_df)}] {status} {qtype} | expected={answer} | predicted={predicted} | tokens={len(gen_ids)}")
            print(f"{'='*60}")
            print(f"  FULL OUTPUT ({len(generated)} chars):")
            print(generated)
            print(f"{'─'*60}")

            eval_results.append({
                'id': row['id'],
                'type': qtype,
                'answer': answer,
                'predicted': predicted,
                'correct': correct,
                'gen_tokens': len(gen_ids),
                'generated': generated,
            })

            # Free KV cache every sample
            del output, inputs, gen_ids
            torch.cuda.empty_cache()

    except Exception as _eval_err:
        print(f"\n⚠️ Eval stopped after {len(eval_results)}/{len(holdout_df)} samples: {_eval_err}")
        print("Adapter was already saved — no data loss.")

    eval_time = time.time() - t0_eval

    # --- Results table (even if partial) ---
    if eval_results:
        import pandas as pd
        results_df = pd.DataFrame(eval_results)

        print(f"\n{'='*60}")
        _label = "PARTIAL" if len(eval_results) < len(holdout_df) else "FULL"
        print(f"  📊 HOLDOUT RESULTS [{_label}] (eval time: {eval_time/60:.1f} min)")
        print(f"{'='*60}")
        print(f"  {'Type':12s} {'Correct':>8s} {'Total':>6s} {'Acc':>8s} {'Avg Tokens':>11s}")
        print(f"  {'-'*47}")

        for t in sorted(results_df['type'].unique()):
            t_df = results_df[results_df['type'] == t]
            n_correct = t_df['correct'].sum()
            n_total = len(t_df)
            avg_tok = t_df['gen_tokens'].mean()
            acc = n_correct / n_total * 100
            bar = '█' * int(acc / 5) + '░' * (20 - int(acc / 5))
            print(f"  {t:12s} {n_correct:5d}/{n_total:<3d}   {acc:5.1f}%  {avg_tok:8.0f}  {bar}")

        total_correct = results_df['correct'].sum()
        total_n = len(results_df)
        total_acc = total_correct / total_n * 100
        print(f"  {'-'*47}")
        print(f"  {'TOTAL':12s} {total_correct:5d}/{total_n:<3d}   {total_acc:5.1f}%")

        # --- Show failures ---
        failures = results_df[~results_df['correct']]
        if len(failures) > 0:
            print(f"\n  ❌ Failures ({len(failures)} total, showing first 10):")
            for _, f in failures.head(10).iterrows():
                print(f"    [{f['type']:10s}] expected='{f['answer']}', got='{f['predicted']}' ({f['gen_tokens']} tok)")

        # --- Token stats ---
        print(f"\n  Token stats: min={results_df['gen_tokens'].min()}, "
              f"median={results_df['gen_tokens'].median():.0f}, "
              f"max={results_df['gen_tokens'].max()}")
        not_found = (results_df['predicted'] == 'NOT_FOUND').sum()
        if not_found > 0:
            print(f"  ⚠️ NOT_FOUND: {not_found} samples (no \\boxed{{}} in output)")
    else:
        print("No eval results collected.")

else:
    print("Holdout evaluation: SKIPPED (HOLDOUT_ENABLED=False)")

# =============================================
#  TRAINING SUMMARY
# =============================================
print("\n" + "=" * 60)
print("  TRAINING SUMMARY")
print("=" * 60)
print(f"  Stage 1: loss={_s1_loss:.4f}, time={stage1_time/60:.1f}min")
if STAGE2_ENABLED:
    print(f"  Stage 2: {n_sample} samples, loss={_s2_loss:.4f}, time={stage2_time/60:.1f}min")
    print(f"  Total training time: {(stage1_time + stage2_time)/60:.1f} min")
else:
    print(f"  Stage 2: SKIPPED")
    print(f"  Total time: {stage1_time/60:.1f} min")
print(f"  LoRA rank: {LORA_RANK}, alpha: {LORA_ALPHA}")
print(f"  Prompt suffix verified: ✅")
print(f"  Output: /kaggle/working/submission.zip")

if HOLDOUT_ENABLED and results_df is not None and len(results_df) > 0:
    total_correct = results_df['correct'].sum()
    total_n = len(results_df)
    print(f"  Holdout: {total_correct}/{total_n} = {total_correct/total_n*100:.1f}%")
    print(f"  Eval time: {eval_time/60:.1f} min")

print("=" * 60)

Freeing training memory...
GPU free memory after cleanup: 32.1 GB
Merging LoRA into base model...


GPU free memory after merge: 35.4 GB (saved 3.3 GB)
Holdout types: ['cipher', 'gravity', 'numeral', 'unit_conv']

  HOLDOUT EVAL: 80 samples
  Types: ['cipher', 'gravity', 'numeral', 'unit_conv']
  Decoding: greedy (temperature=0.0) -- matches official eval
  Max new tokens: 512


/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/torch/utils/checkpoint.py:235: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)



  [1/80] ❌ cipher | expected=the golden alice discovers | predicted=the bright alice discovers | tokens=141
  FULL OUTPUT (328 chars):
This is a substitution cipher.

Mapping (relevant chars): d→r, e→a, f→o, g→v, h→n, i→l, j→s, n→i, p→c, t→d, u→h, x→t, y→p, z→e

Decrypting 'xuz yfitzh einpz tnjpfgzdj':
  xuz → the
  yfitzh → bright
  einpz → alice
  tnjpfgzdj → discovers

Result: the bright alice discovers
</think>
\boxed{the bright alice discovers}<|im_end|>
────────────────────────────────────────────────────────────

  [2/80] ✅ cipher | expected=teacher writes mirror | predicted=teacher writes mirror | tokens=113
  FULL OUTPUT (284 chars):
This is a substitution cipher.

Mapping (relevant chars): a→c, c→o, g→a, j→m, l→h, m→i, n→s, r→t, s→r, w→e, x→w

Decrypting 'rwgalws xsmrwn jmsscs':
  rwgalws → teacher
  xsmrwn → writes
  jmsscs → mirror

Result: teacher writes mirror
</think>
\boxed{teacher writes mirror}<|im_end|>
────────────────────────────────────────────────────────────

 